# 第 08 课 · 采样、探索与奖励漏洞

前面的 RL 依赖“生成多个答案，再给奖励”。这节把采样与打分拆开看：温度如何影响答案分布？为什么“包含正确数字”不是可靠奖励？

本课只做推理，不更新权重。加载第四课不充分训练的基线，以便观察多样性。固定 5 道题，每题每种温度采样 64 次；这些是诊断样例，不用于宣称测试性能。

## 1 · 温度与采样

温度 T 用 logits/T 改变采样分布。较高温度通常增加多样性，但不保证提升准确率。贪心解码直接取最大概率 token，不等同于某个非零温度的随机采样。

比较温度 0.5、1.0、1.5；输出上限均为 4 token。EOS 后字符不计入答案。

In [ ]:
import json, torch, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from transformers import Qwen2ForCausalLM
assert torch.cuda.is_available()
torch.set_num_threads(2)
root=Path('experiment04_output')
info=json.loads((root/'results.json').read_text())
vocab=info['vocab'];stoi={c:i for i,c in enumerate(vocab)}
model=Qwen2ForCausalLM.from_pretrained(root/'baseline').cuda().eval().requires_grad_(False)
@torch.inference_mode()
def sample(a,b,temperature,count=64):
    prompt=f'{a}+{b}='
    x=torch.tensor([[stoi[c] for c in prompt]],device='cuda').repeat(count,1)
    for _ in range(4):
        logits=model(input_ids=x,use_cache=False).logits[:,-1]/temperature
        token=torch.distributions.Categorical(logits=logits).sample()
        x=torch.cat([x,token[:,None]],dim=1)
    raw=[''.join(vocab[t] for t in row) for row in x[:,len(prompt):].tolist()]
    return [(r.split('#')[0],'#' in r) for r in raw]
rows=[]
for temperature in [0.5,1.0,1.5]:
    torch.manual_seed(2029);torch.cuda.manual_seed_all(2029)
    for a,b in [(3,7),(12,7),(19,19),(8,9),(15,6)]:
        for answer,ended in sample(a,b,temperature):
            rows.append(dict(temperature=temperature,prompt=f'{a}+{b}=',answer=answer,
                target=str(a+b),ended=ended,correct=ended and answer==str(a+b)))
predictions=pd.DataFrame(rows)
per_question=predictions.groupby(['temperature','prompt']).agg(accuracy=('correct','mean'),distinct=('answer','nunique')).reset_index()
summary=per_question.groupby('temperature')[['accuracy','distinct']].mean()
display(summary)
summary.plot(subplots=True,marker='o',figsize=(7,5),title=['Sample accuracy','Mean distinct answers per question'])
plt.tight_layout();plt.show()
display(predictions.groupby(['temperature','prompt','answer']).size().rename('count').reset_index().head(20))

## 2 · 奖励漏洞：得分高不等于回答好

若奖励只检查回答中是否出现目标字符串，`120`、`12 或 99` 甚至列举所有数字都可能被误判为正确。这是奖励函数设计问题，不代表模型能力提升。

下面是人工构造的攻击样例，**不是声称训练模型自动发现了漏洞**。本教程的纯数字任务要求规范整数字符串；如果将来允许自然语言解释，应重新定义输出协议与解析规则，不能直接照搬严格匹配。

In [ ]:
def loose_reward(answer,target):return float(str(target) in answer)
def strict_reward(answer,target):return float(answer.strip()==str(target))
cases=[('12',1),('120',0),('答案是 12 或 99',0),('012',0),('',0),(' 12 ',1),(' '.join(map(str,range(200))),0)]
audit=pd.DataFrame([dict(answer=s if len(s)<50 else s[:50]+'…',expected=e,loose=loose_reward(s,12),strict=strict_reward(s,12)) for s,e in cases])
assert all(strict_reward(s,12)==e for s,e in cases)
display(audit)
print('宽松奖励误判数量:',sum(loose_reward(s,12)!=e for s,e in cases))
out=Path('experiment08_output');out.mkdir(exist_ok=True)
predictions.to_csv(out/'sampling.csv',index=False);audit.to_csv(out/'reward_audit.csv',index=False)
summary.to_csv(out/'summary.csv')

## 3 · 小结与练习

GRPO 的组大小、温度和奖励函数共同影响有效学习信号。只增加采样数并不能修复错误奖励。

练习：为 `-12`、`12.0`、多行文本、空白字符各写一个测试，并先说明是否应该接受。用业务目标定义预期，再实现奖励。

这节结果只涵盖固定诊断题；若要比较解码策略，应预先规定完整评估集、随机种子和采样预算。